In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from sklearn.isotonic import IsotonicRegression
from scipy.signal import savgol_filter

# --------------------------
# 1) Detect rest periods
# --------------------------
def find_rest_periods(df, current_col="batt_current", rest_current_thresh=0.1, min_rest_s=3600):
    """
    Return list of (start_idx, end_idx) where |current| <= rest_current_thresh
    for at least min_rest_s seconds.
    """
    cur = df[current_col].abs().to_numpy()
    t = df["time"].to_numpy()
    is_rest = cur <= rest_current_thresh

    # find runs of True
    padded = np.concatenate(([False], is_rest, [False]))
    changes = np.diff(padded.astype(int))
    starts = np.where(changes == 1)[0]
    ends = np.where(changes == -1)[0] - 1

    rest_periods = []
    for s, e in zip(starts, ends):
        duration = t[e] - t[s]
        if duration >= min_rest_s:
            rest_periods.append((s, e))
    return rest_periods


# --------------------------
# 2) Collect OCV samples from rests
# --------------------------
def collect_ocv_samples_from_rests(df, rest_periods, cell_prefix="cell_voltages_", soc_col="soc",
                                   temp_col=None, sample_method="mean"):
    """
    For each rest period, compute representative SOC and representative cell voltages.
    Returns DataFrame with rows per rest: columns: soc, temp (optional), cell_voltages_*.
    sample_method: 'mean' or 'median'
    """
    all_samples = []
    cell_cols = [c for c in df.columns if c.startswith(cell_prefix)]
    for (s, e) in rest_periods:
        seg = df.iloc[s:e+1]
        if sample_method == "median":
            soc_val = seg[soc_col].median()
            volt_vals = seg[cell_cols].median(axis=0)
            temp_val = seg[temp_col].median() if temp_col and temp_col in seg.columns else None
        else:
            soc_val = seg[soc_col].mean()
            volt_vals = seg[cell_cols].mean(axis=0)
            temp_val = seg[temp_col].mean() if temp_col and temp_col in seg.columns else None

        row = {"soc": float(soc_val)}
        if temp_val is not None:
            row["temp"] = float(temp_val)
        for c in cell_cols:
            row[c] = float(volt_vals[c])
        all_samples.append(row)

    samples_df = pd.DataFrame(all_samples)
    return samples_df


# --------------------------
# 3) Fit OCV curve per cell: bin SOC and compute robust median, isotonic regression, smoothing
# --------------------------
def fit_ocv_from_samples(samples_df, cell_prefix="cell_voltages_", soc_bins=np.arange(0, 100.1, 1.0),
                         smoothing_window=11, smoothing_poly=3):
    """
    Fit OCV(SOC) for each cell from collected rest samples.
    Returns:
      ocv_funcs: dict cell_col -> interp1d(soc -> voltage)
      ocv_inv_funcs: dict cell_col -> interp1d(voltage -> soc)
      ocv_table: DataFrame of binned medians for inspection
    """
    from sklearn.isotonic import IsotonicRegression

    cell_cols = [c for c in samples_df.columns if c.startswith(cell_prefix)]
    soc_vals = samples_df["soc"].to_numpy()

    # create soc bin centers
    bin_edges = soc_bins
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    ocv_table = pd.DataFrame({"soc_bin_center": bin_centers})
    ocv_funcs = {}
    ocv_inv_funcs = {}

    for c in cell_cols:
        # compute median voltage per SOC bin
        medians = []
        for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
            mask = (soc_vals >= lo) & (soc_vals < hi)
            if mask.sum() == 0:
                medians.append(np.nan)
            else:
                medians.append(np.nanmedian(samples_df.loc[mask, c].to_numpy()))
        medians = np.array(medians, dtype=float)

        # fill NaNs by interpolation across bins (1D)
        finite = np.isfinite(medians)
        if finite.sum() < 2:
            # not enough samples to fit a curve; fallback to raw samples interpolation
            xs = soc_vals
            ys = samples_df[c].to_numpy()
            # sort by soc
            order = np.argsort(xs)
            xs = xs[order]; ys = ys[order]
            # isotonic to impose monotonicity
            ir = IsotonicRegression(out_of_bounds='clip')
            try:
                y_iso = ir.fit_transform(xs, ys)
                ocv_func = interp1d(xs, y_iso, bounds_error=False, fill_value="extrapolate")
                ocv_funcs[c] = ocv_func
                ocv_inv_funcs[c] = interp1d(y_iso, xs, bounds_error=False, fill_value=(xs.min(), xs.max()))
            except Exception:
                ocv_funcs[c] = interp1d(xs, ys, bounds_error=False, fill_value="extrapolate")
                ocv_inv_funcs[c] = interp1d(ys, xs, bounds_error=False, fill_value=(xs.min(), xs.max()))
            continue

        # interpolate missing bins (linear)
        med_interp = np.copy(medians)
        xi = bin_centers[finite]
        yi = medians[finite]
        med_interp[~finite] = np.interp(bin_centers[~finite], xi, yi)

        # apply isotonic regression to enforce monotonic increasing OCV(SOC)
        ir = IsotonicRegression(out_of_bounds='clip')
        soc_for_ir = bin_centers
        try:
            med_iso = ir.fit_transform(soc_for_ir, med_interp)
        except Exception:
            med_iso = med_interp

        # optional smoothing with Savitzky-Golay (window must be odd and <= len)
        if smoothing_window is not None and smoothing_window > 1 and len(med_iso) >= smoothing_window:
            try:
                med_smooth = savgol_filter(med_iso, smoothing_window if smoothing_window % 2 == 1 else smoothing_window+1,
                                           polyorder=smoothing_poly, mode="interp")
            except Exception:
                med_smooth = med_iso
        else:
            med_smooth = med_iso

        ocv_table[c + "_median"] = medians
        ocv_table[c + "_interp"] = med_interp
        ocv_table[c + "_iso"] = med_iso
        ocv_table[c + "_smooth"] = med_smooth

        # build interpolation functions
        ocv_func = interp1d(bin_centers, med_smooth, kind="linear", bounds_error=False, fill_value="extrapolate")
        # invert monotonic function to get OCV_inv: voltage -> soc
        # ensure monotonic increasing in med_smooth, else fallback
        try:
            ocv_inv_func = interp1d(med_smooth, bin_centers, kind="linear",
                                     bounds_error=False, fill_value=(bin_centers[0], bin_centers[-1]))
        except Exception:
            ocv_inv_func = interp1d(bin_centers, med_smooth, kind="linear", bounds_error=False,
                                     fill_value="extrapolate")
        ocv_funcs[c] = ocv_func
        ocv_inv_funcs[c] = ocv_inv_func

    return ocv_funcs, ocv_inv_funcs, ocv_table


# --------------------------
# 4) Estimate per-cell capacity using OCV_inv and discharge integrals
# --------------------------
def estimate_cell_capacity_from_ocv(df, ocv_inv_funcs, cell_prefix="cell_voltages_",
                                   soc_col="soc", current_col="batt_current"):
    """
    For each discharge segment, compute pack Ah removed Q (Ah),
    compute per-cell SOC start/end using ocv_inv_funcs and cell voltages at start/end rest,
    and derive capacity estimate cap_i = Q / (delta_SOC_i/100).
    Returns DataFrame cap_estimates with one row per discharge and columns per cell.
    """
    segments = find_discharge_segments(df)
    cell_cols = [c for c in df.columns if c.startswith(cell_prefix)]
    cap_rows = []
    for seg_id, (s, e) in enumerate(segments, 1):
        seg = df.iloc[s:e+1].reset_index(drop=True)
        if len(seg) < 2:
            continue

        # pack Q removed (Ah) over this segment: integrate current (negative for discharge)
        time_s = seg["time"].to_numpy()
        I = seg[current_col].to_numpy()
        # integrate as sum(I * dt) / 3600 -> Ah; use trapezoid
        dt = np.diff(time_s, prepend=time_s[0])
        Q_Ah = -np.sum(I * dt) / 3600.0  # negative current during discharge -> make positive Ah removed

        # use first and last *rest* voltages if available;
        # fallback to first/last sample in the segment
        V_start = seg.loc[0, cell_cols].to_numpy()
        V_end = seg.loc[len(seg)-1, cell_cols].to_numpy()

        # map voltages to SOC via ocv_inv_funcs (per cell)
        soc_start_cells = []
        soc_end_cells = []
        for c, V0, V1 in zip(cell_cols, V_start, V_end):
            invf = ocv_inv_funcs.get(c)
            if invf is None:
                soc_start_cells.append(np.nan)
                soc_end_cells.append(np.nan)
            else:
                soc_start_cells.append(float(invf(V0)))
                soc_end_cells.append(float(invf(V1)))

        soc_start_cells = np.array(soc_start_cells, dtype=float)
        soc_end_cells = np.array(soc_end_cells, dtype=float)
        delta_soc = soc_start_cells - soc_end_cells  # percent points

        # Avoid divide by zero: if delta_soc very small, skip this discharge
        small_mask = np.abs(delta_soc) < 0.5  # require at least 0.5% change to use
        if np.all(small_mask):
            continue

        # capacity estimate per cell: Q_Ah / (delta_soc/100)
        cap_est = np.full_like(delta_soc, np.nan, dtype=float)
        valid = ~small_mask
        cap_est[valid] = Q_Ah / (delta_soc[valid] / 100.0)

        row = {"segment_id": seg_id, "Q_Ah": Q_Ah}
        for c, cap in zip(cell_cols, cap_est):
            row[c + "_cap_Ah"] = float(cap) if np.isfinite(cap) else np.nan
        cap_rows.append(row)

    cap_df = pd.DataFrame(cap_rows)
    return cap_df


# --------------------------
# 5) Aggregate capacities into SOH per cell and flag outliers
# --------------------------
def aggregate_soh_from_cap_df(cap_df, nominal_capacity_Ah=116.0, defect_threshold_frac=0.10):
    """
    cap_df has rows per discharge with columns cellX_cap_Ah.
    Compute mean capacity per cell (median recommended), compute SOH = cap / nominal_capacity,
    and flag cells with SOH <= (1 - defect_threshold_frac) * mean(SOH_all).
    """
    cap_cols = [c for c in cap_df.columns if c.endswith("_cap_Ah")]
    if len(cap_cols) == 0:
        return pd.DataFrame(), []

    # median across discharges is robust
    cap_median = cap_df[cap_cols].median(skipna=True)
    soh = cap_median / nominal_capacity_Ah
    mean_soh = np.nanmean(soh)
    flags = soh <= (1.0 - defect_threshold_frac) * mean_soh

    results = pd.DataFrame({
        "cell": [c.replace("_cap_Ah", "").replace("_cap_Ah", "") for c in cap_cols],
        "cap_median_Ah": cap_median.values,
        "soh": soh.values,
        "flag_low_soh": flags.astype(bool)
    })
    return results, results.loc[results["flag_low_soh"], "cell"].tolist()


# --------------------------
# 6) Full pipeline wrapper
# --------------------------
def ocv_derive_and_soh_estimate(df,
                                cell_prefix="cell_voltages_",
                                soc_col="soc",
                                current_col="batt_current",
                                rest_current_thresh=0.1,
                                min_rest_s=3600,
                                soc_bins=np.arange(0, 100.1, 1.0),
                                nominal_capacity_Ah=116.0,
                                defect_threshold_frac=0.10):
    # 1) find rest periods
    rests = find_rest_periods(df, current_col=current_col,
                              rest_current_thresh=rest_current_thresh, min_rest_s=min_rest_s)
    if len(rests) == 0:
        raise RuntimeError("No rest periods found with given thresholds; cannot derive OCV.")

    # 2) collect OCV samples
    samples = collect_ocv_samples_from_rests(df, rests, cell_prefix=cell_prefix, soc_col=soc_col)

    # 3) fit ocv curves
    ocv_funcs, ocv_inv_funcs, ocv_table = fit_ocv_from_samples(samples, cell_prefix, soc_bins)

    # 4) estimate capacities per discharge using ocv_inv
    cap_df = estimate_cell_capacity_from_ocv(df, ocv_inv_funcs, cell_prefix=cell_prefix,
                                             soc_col=soc_col, current_col=current_col)

    # 5) aggregate SOH and flag
    soh_results, flagged_cells = aggregate_soh_from_cap_df(cap_df, nominal_capacity_Ah, defect_threshold_frac)

    return {
        "rests": rests,
        "samples": samples,
        "ocv_funcs": ocv_funcs,
        "ocv_inv_funcs": ocv_inv_funcs,
        "ocv_table": ocv_table,
        "cap_df": cap_df,
        "soh_results": soh_results,
        "flagged_cells": flagged_cells
    }


In [1]:
# ----- PARAMETERS ----- #
INPUT_FOLDER = r"C:\Users\mmackenzie\OneDrive - ZELEROS GLOBAL S.L\Documents\Synthetic data\Yearly profiles"
BATTERY_ID = "battery_03"
DELTA_I_THRESH = 20              # A; threshold to detect a current step (transient)
MIN_STEP_DT_S = 60                # minimum time diff (s) for transient measurement
RESISTANCE_OUTLIER_FACTOR = 1.2     # R_i > mean_R * factor => flag
CAPACITY_DEFECT_FACTOR = 0.90     # capacity < mean_capacity * factor => flag
OUTLIER_PERSISTANCE_LIMIT = 10    # How long the outlier should persist before being flagged as an anomaly
IMBALANCE_V_THRESH = 0.1          # V; persistent mean deviation threshold
IMBALANCE_PERSIST_H = 24.0        # hours; how long deviation must persist to count
N_CELLS = 13
N_TEMPS = 5

# Nominal pack data
NOMINAL_CAPACITY_AH = 116*4    # nominal cell capacity Ah

In [2]:
input_file = os.path.join(INPUT_FOLDER, BATTERY_ID + ".csv")
df = pd.read_csv(input_file)

NameError: name 'os' is not defined